In [6]:
# Rode esta célula uma vez. Se ela instalar/atualizar algo, reinicie o kernel (Restart) antes de importar.
%pip install langchain==1.4.7 langchain-community==0.3.31 langchain-text-splitters==1.1.3 langchain-classic==1.0.8 langchain-google-genai==4.4.2 langchain-chroma==1.1.0 chromadb pypdf fonttools python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [7]:
# Importações básicas
import os

# Carrega a GOOGLE_API_KEY do arquivo .env
from dotenv import load_dotenv
load_dotenv(override=True)  # relê o .env a cada execução

# loader de documentos PDF
from langchain_community.document_loaders import PyPDFLoader

# Divisão de texto em blocos
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings (Google Gemini - tier gratuito)
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Banco vetorial
from langchain_chroma import Chroma

# LLM (Google Gemini - tier gratuito)
from langchain_google_genai import ChatGoogleGenerativeAI

# Cadeia RAG
from langchain_classic.chains import RetrievalQA

 LER O PDF COM AS REGRAS DE FUTEBOL


In [8]:
caminho_pdf = "docs/regras-futebol.pdf"
__loader__ = PyPDFLoader(caminho_pdf)
docs = __loader__.load()
print(f"{len(docs)} páginas lidas.")

232 páginas lidas.


Divisão do documento em chunks, pequenos blocos para facilitar a recuperação de contexto.

In [9]:
# divisão em chunks menores
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # tamanho do corte
    chunk_overlap=100,
)

chunks = text_splitter.split_documents(docs)
print(len(chunks))

654


Cada bloco será convertido para vetores semânticos e armazenados em um banco vetorial

In [ ]:
load_dotenv(override=True)

# O .env usa GOOGLE_API_KEY (GEMINI_API_KEY também é aceito)
api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("Preencha GOOGLE_API_KEY no arquivo .env")

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",  # nome correto do modelo
    api_key=api_key,
)

# Abre (ou cria) o banco vetorial persistente
vector_store = Chroma(
    collection_name="regras_futebol",
    embedding_function=embeddings,
    persist_directory="./chroma_regras_futebol",
)

print("Blocos já indexados:", len(vector_store.get()["ids"]))

### Indexação no banco vetorial
Cada bloco é convertido em vetor e gravado no Chroma. O tier gratuito do Gemini limita a cerca de 100 blocos por minuto, então a indexação é feita em lotes. Se a célula for interrompida, basta rodá-la novamente: ela continua de onde parou.

In [ ]:
import re
import time

LOTE = 80    # o tier gratuito do Gemini aceita ~100 blocos por minuto
PAUSA = 65   # segundos entre lotes para não estourar a cota

ja_indexados = len(vector_store.get()["ids"])
if ja_indexados >= len(chunks):
    print(f"Banco já completo: {ja_indexados} blocos.")
else:
    for inicio in range(ja_indexados, len(chunks), LOTE):
        lote = chunks[inicio:inicio + LOTE]
        while True:
            try:
                vector_store.add_documents(lote)
                break
            except Exception as erro:
                if "RESOURCE_EXHAUSTED" not in str(erro):
                    raise
                # a API costuma informar o tempo de espera (ex.: "retry in 12.3s")
                espera = PAUSA
                achou = re.search(r"retry in ([\d.]+)s", str(erro))
                if achou:
                    espera = float(achou.group(1)) + 2
                print(f"Limite de requisições atingido. Aguardando {espera:.0f}s...")
                time.sleep(espera)
        print(f"Indexados: {min(inicio + LOTE, len(chunks))}/{len(chunks)}")
        if inicio + LOTE < len(chunks):
            time.sleep(PAUSA)

print("Total de blocos no banco:", len(vector_store.get()["ids"]))

### Consulta (RAG)
A pergunta é usada para buscar os trechos mais parecidos no banco vetorial. Esses trechos são enviados ao LLM, que responde com base nas regras do PDF.

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",  # use "gemini-flash-lite-latest" se necessário
    api_key=api_key,
)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vector_store.as_retriever(search_kwargs={"k": 4}),
    return_source_documents=True,
)

In [ ]:
pergunta = "Quantos jogadores tem um time de futebol em campo?"

resposta = qa.invoke({"query": pergunta})
print(resposta["result"])

print("\nTrechos consultados:")
for doc in resposta["source_documents"]:
    print(f"- página {doc.metadata.get('page', 0) + 1}")